# BRBseq and WGS sample ID QC

Check identity matches from the combined gill/WGS and liver/WGS relatedness outputs, then compare independent WGS, gill BRB, and liver BRB ancestry results against recorded population and ecotype.

Analysis IDs end in _D (WGS), _G (gill), or _L (liver). ADMIXTURE component labels are independent between runs, so Q1 in one dataset is not assumed to correspond to Q1 in another.

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(readr)

find_repo_root <- function(start = getwd()) {
  candidates <- normalizePath(c(start, file.path(start, '..')), mustWork = FALSE)
  hit <- candidates[file.exists(file.path(candidates, 'config', 'brb_pipeline_config.sh'))]
  if (!length(hit)) stop('Could not find Gacu_BC_txmics repo root from: ', start)
  hit[[1]]
}
repo_root <- find_repo_root()
genotype_root <- file.path(repo_root, 'outputs', 'genotypes')
plink_root <- file.path(genotype_root, 'plink')
relatedness_root <- file.path(genotype_root, 'relatedness')
mapping_file <- file.path(repo_root, 'metadata', 'derived', 'sample_source_mapping.tsv')
stopifnot(file.exists(mapping_file))
sample_map <- read_tsv(mapping_file, show_col_types = FALSE) %>%
  distinct(analysis_id, .keep_all = TRUE) %>%
  mutate(base_id = sub('_[DGL]$', '', analysis_id))

# Review CV error first, then set K for the ancestry summaries.
selected_k <- 8L
k_values <- 8:28
datasets <- c(wgs = 'wgs', gill = 'gill_brb', liver = 'liver_brb')
cat('Repo:', repo_root, '\nMapped analysis IDs:', nrow(sample_map), '\n')
sample_map %>% count(source, tissue)

## Identity matching from combined-panel relatedness

For every BRB sample, compare PI_HAT to the WGS ID with the same base ID against the strongest alternative WGS match. An expected match that ranks first and exceeds alternatives supports the recorded identity. An alternative match that ranks first is a candidate to investigate, not an automatic relabeling.

In [ ]:
read_identity_comparison <- function(tissue) {
  path <- file.path(relatedness_root, paste0(tissue, '_wgs_brb_relatedness.genome'))
  if (!file.exists(path)) stop('Missing relatedness output: ', path)
  brb_suffix <- if (tissue == 'gill') 'G' else 'L'
  dat <- read.table(path, header = TRUE, check.names = FALSE, stringsAsFactors = FALSE) %>%
    mutate(suffix1 = sub('.*_', '', IID1), suffix2 = sub('.*_', '', IID2)) %>%
    filter((suffix1 == 'D' & suffix2 == brb_suffix) |
           (suffix2 == 'D' & suffix1 == brb_suffix)) %>%
    transmute(
      brb_id = if_else(suffix1 == brb_suffix, IID1, IID2),
      wgs_id = if_else(suffix1 == 'D', IID1, IID2),
      PI_HAT, Z0, Z1, Z2, IBS0, PPC,
      expected_pair = sub('_[DGL]$', '', brb_id) == sub('_[DGL]$', '', wgs_id)
    ) %>%
    group_by(brb_id) %>%
    mutate(wgs_rank = min_rank(desc(PI_HAT))) %>%
    ungroup()
  expected <- dat %>% filter(expected_pair) %>%
    transmute(brb_id, expected_wgs_id = wgs_id, expected_PI_HAT = PI_HAT, expected_rank = wgs_rank)
  alternative <- dat %>% filter(!expected_pair) %>% group_by(brb_id) %>%
    slice_max(PI_HAT, n = 1, with_ties = FALSE) %>% ungroup() %>%
    transmute(brb_id, best_alternative_wgs_id = wgs_id, best_alternative_PI_HAT = PI_HAT)
  brb_ids <- sample_map %>% filter(source == 'BRB', tissue == !!tissue) %>%
    select(brb_id = analysis_id)
  brb_ids %>% left_join(expected, by = 'brb_id') %>%
    left_join(alternative, by = 'brb_id') %>%
    mutate(tissue = tissue, expected_minus_alternative = expected_PI_HAT - best_alternative_PI_HAT) %>%
    left_join(sample_map %>% select(brb_id = analysis_id, population, ecotype), by = 'brb_id') %>%
    arrange(expected_rank, desc(expected_PI_HAT))
}
identity_qc <- bind_rows(read_identity_comparison('gill'), read_identity_comparison('liver'))
identity_qc %>% count(tissue, has_expected_pair = !is.na(expected_PI_HAT), expected_is_top_match = expected_rank == 1)
identity_qc %>% select(tissue, brb_id, population, ecotype, expected_wgs_id, expected_PI_HAT, expected_rank,
                        best_alternative_wgs_id, best_alternative_PI_HAT, expected_minus_alternative)

In [ ]:
ggplot(identity_qc, aes(expected_PI_HAT, best_alternative_PI_HAT, color = population)) +
  geom_abline(slope = 1, intercept = 0, linetype = 'dashed', color = 'grey50') +
  geom_point(alpha = 0.75) + facet_wrap(~ tissue) + coord_equal() + theme_bw() +
  labs(x = 'PI_HAT to expected same-ID WGS', y = 'PI_HAT to strongest alternative WGS',
       title = 'Expected identity match versus strongest alternative')

## Independent ADMIXTURE results

The CV plot reports results as they become available for K=8–28. Choose K for each dataset after reviewing those curves. For the selected K, Q rows are matched to sample IDs in the corresponding PLINK FAM file; row order must agree. Population summaries and sample flags are calculated separately within each dataset.

In [ ]:
read_cv_error <- function(panel, k) {
  path <- file.path(genotype_root, paste0(panel, '_admixture_K', k, '.log'))
  if (!file.exists(path)) return(NULL)
  cv_line <- grep('CV error', readLines(path, warn = FALSE), value = TRUE)
  if (!length(cv_line)) return(NULL)
  value <- suppressWarnings(as.numeric(sub('.*: *', '', cv_line[[length(cv_line)]])))
  if (!is.finite(value)) return(NULL)
  tibble(dataset = names(datasets)[match(panel, datasets)], K = k, CV_error = value)
}
cv_results <- bind_rows(lapply(unname(datasets), function(panel) {
  bind_rows(lapply(k_values, function(k) read_cv_error(panel, k)))
}))
if (nrow(cv_results)) {
  ggplot(cv_results, aes(K, CV_error, color = dataset)) + geom_line() + geom_point() + theme_bw() +
    labs(title = 'ADMIXTURE cross-validation error', y = 'CV error')
} else message('No ADMIXTURE CV logs found yet.')

read_admixture <- function(dataset, panel, k = selected_k) {
  prefix <- file.path(plink_root, paste0(panel, '_ldpruned'))
  fam_path <- paste0(prefix, '.fam')
  q_path <- paste0(prefix, '.', k, '.Q')
  if (!file.exists(fam_path) || !file.exists(q_path)) {
    message('Skipping ', dataset, ': expected FAM/Q input is not present yet.')
    return(NULL)
  }
  fam <- read.table(fam_path, stringsAsFactors = FALSE,
                    col.names = c('FID', 'IID', 'father', 'mother', 'sex', 'phenotype'))
  q <- read.table(q_path, stringsAsFactors = FALSE, check.names = FALSE)
  if (nrow(fam) != nrow(q)) stop('FAM/Q row mismatch for ', dataset)
  names(q) <- paste0('Q', seq_len(ncol(q)))
  bind_cols(tibble(dataset = dataset, analysis_id = fam$IID), as_tibble(q)) %>%
    left_join(sample_map %>% select(analysis_id, base_id, source, tissue, population, ecotype),
              by = 'analysis_id')
}
ancestry <- bind_rows(lapply(names(datasets), function(dataset) {
  read_admixture(dataset, unname(datasets[[dataset]]))
}))
if (!nrow(ancestry)) stop('No selected-K ADMIXTURE results were found.')
q_columns <- grep('^Q[0-9]+$', names(ancestry), value = TRUE)
ancestry <- ancestry %>% rowwise() %>%
  mutate(top_component = q_columns[which.max(c_across(all_of(q_columns)))],
         top_ancestry = max(c_across(all_of(q_columns)), na.rm = TRUE)) %>% ungroup()
ancestry %>% count(dataset, source, tissue, population, ecotype, name = 'n_samples')

In [ ]:
population_ancestry <- ancestry %>% filter(!is.na(population)) %>%
  pivot_longer(all_of(q_columns), names_to = 'component', values_to = 'proportion') %>%
  group_by(dataset, population, ecotype, component) %>%
  summarize(mean_proportion = mean(proportion, na.rm = TRUE), .groups = 'drop')
ggplot(population_ancestry, aes(interaction(population, ecotype, sep = ' / '),
                              mean_proportion, fill = component)) + geom_col() +
  facet_wrap(~ dataset, ncol = 1) + theme_bw() +
  theme(axis.text.x = element_text(angle = 60, hjust = 1)) +
  labs(x = 'Recorded population / ecotype', y = 'Mean ancestry proportion',
       title = paste('Population ancestry summaries at K =', selected_k))

population_modes <- ancestry %>% filter(!is.na(population), !is.na(top_component)) %>%
  count(dataset, population, ecotype, top_component, name = 'n') %>%
  group_by(dataset, population, ecotype) %>% slice_max(n, n = 1, with_ties = FALSE) %>%
  mutate(modal_fraction = n / sum(n)) %>% ungroup() %>%
  rename(population_modal_component = top_component)
ancestry_flags <- ancestry %>%
  left_join(population_modes %>% select(dataset, population, ecotype,
                                         population_modal_component, modal_fraction),
            by = c('dataset', 'population', 'ecotype')) %>%
  filter(!is.na(population_modal_component), top_ancestry >= 0.60,
         top_component != population_modal_component) %>%
  arrange(dataset, population, ecotype, desc(top_ancestry)) %>%
  select(dataset, analysis_id, base_id, population, ecotype, top_component, top_ancestry,
         population_modal_component, modal_fraction)
ancestry_flags

## Compare ancestry for the same fish across datasets

The next cell aligns component labels between WGS and each tissue using correlations across fish whose expected same-ID pair ranks first in relatedness. It then reports per-fish profile differences and whether the dominant component agrees after alignment. Review these flags with relatedness margin and population metadata.

In [ ]:
compare_ancestry_to_wgs <- function(tissue, tissue_dataset) {
  wgs <- ancestry %>% filter(dataset == 'wgs') %>% select(base_id, all_of(q_columns))
  brb <- ancestry %>% filter(dataset == tissue_dataset) %>% select(base_id, analysis_id, all_of(q_columns))
  w_names <- paste0('W', seq_along(q_columns)); b_names <- paste0('B', seq_along(q_columns))
  names(wgs)[match(q_columns, names(wgs))] <- w_names
  names(brb)[match(q_columns, names(brb))] <- b_names
  supported <- identity_qc %>% filter(tissue == !!tissue, expected_rank == 1) %>%
    transmute(base_id = sub('_[DGL]$', '', brb_id), relatedness_margin = expected_minus_alternative)
  paired <- inner_join(wgs, brb, by = 'base_id') %>% left_join(supported, by = 'base_id') %>%
    filter(!is.na(relatedness_margin))
  if (nrow(paired) < 3) { message('Too few supported pairs for ', tissue); return(NULL) }
  cors <- cor(as.matrix(paired[, w_names]), as.matrix(paired[, b_names]), use = 'pairwise.complete.obs')
  cors[!is.finite(cors)] <- -Inf
  remain_w <- seq_along(w_names); remain_b <- seq_along(b_names)
  alignment <- vector('list', length(w_names))
  for (i in seq_along(w_names)) {
    subcors <- cors[remain_w, remain_b, drop = FALSE]
    best <- which(subcors == max(subcors), arr.ind = TRUE)[1, ]
    wi <- remain_w[best[1]]; bi <- remain_b[best[2]]
    alignment[[i]] <- tibble(wgs_component = w_names[wi], brb_component = b_names[bi], correlation = cors[wi, bi])
    remain_w <- setdiff(remain_w, wi); remain_b <- setdiff(remain_b, bi)
  }
  alignment <- bind_rows(alignment)
  delta <- sapply(seq_along(w_names), function(i) paired[[alignment$wgs_component[i]]] - paired[[alignment$brb_component[i]]])
  w_top <- w_names[max.col(as.matrix(paired[, w_names]), ties.method = 'first')]
  b_top <- b_names[max.col(as.matrix(paired[, b_names]), ties.method = 'first')]
  mapped_b_top <- alignment$brb_component[match(w_top, alignment$wgs_component)]
  samples <- tibble(dataset = tissue_dataset, base_id = paired$base_id, brb_id = paired$analysis_id,
                    relatedness_margin = paired$relatedness_margin,
                    dominant_component_agrees = mapped_b_top == b_top,
                    mean_absolute_Q_difference = rowMeans(abs(delta))) %>% arrange(desc(mean_absolute_Q_difference))
  list(alignment = mutate(alignment, tissue = tissue), samples = samples)
}
comparisons <- Filter(Negate(is.null), list(compare_ancestry_to_wgs('gill', 'gill'),
                                               compare_ancestry_to_wgs('liver', 'liver')))
component_alignment <- bind_rows(lapply(comparisons, function(x) x[['alignment']]))
sample_ancestry_comparison <- bind_rows(lapply(comparisons, function(x) x[['samples']]))
component_alignment
sample_ancestry_comparison